<a href="https://colab.research.google.com/github/Jordy-20035/ai-reliability-dashboard/blob/main/notebooks/credit_card_temporal_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit card fraud — temporal splits & drift (thesis experiments)

**Purpose (A + B)**
- **(A)** Quantitative experiments: time-ordered splits **D1 → D2 → D3**, drift statistics (PSI, KS) between **D1 vs D2**, classification metrics before/after adaptive retrain (train on **D1∪D2**, evaluate on **D3**).
- **(B)** Export **`D1.csv`**, **`D2.csv`**, **`D3.csv`** for optional use in the main platform (e.g. point **`incoming_csv`** at **`D2.csv`** for a drift check on the “production window”).

**Dataset:** Kaggle *Credit Card Fraud Detection* (`creditcard.csv`): columns `Time`, `V1`…`V28`, `Amount`, `Class`.

**Colab:** Runtime → Run all. Upload `creditcard.csv` when prompted, or set `DATA_PATH` to a Drive path.

**Local:** Place `creditcard.csv` next to this notebook or set `DATA_PATH`.

In [2]:
# Optional: Colab — install deps (uncomment if needed)
!pip -q install pandas numpy scipy scikit-learn

In [6]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RNG = 42
np.random.seed(RNG)

# --- Load data: Colab upload OR path ---
DATA_PATH = "/creditcard.csv"

try:
    from google.colab import files as colab_files

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path(DATA_PATH).is_file():
    print("Upload creditcard.csv")
    uploaded = colab_files.upload()
    name = next(iter(uploaded.keys()))
    Path(name).write_bytes(uploaded[name])
    DATA_PATH = name

df = pd.read_csv(DATA_PATH)
assert "Time" in df.columns and "Class" in df.columns, "Expected creditcard.csv schema"
df = df.sort_values("Time").reset_index(drop=True)
print("Rows:", len(df), "Fraud rate:", df["Class"].mean())

Rows: 284807 Fraud rate: 0.001727485630620034


## 1. Temporal windows D1, D2, D3 (by `Time` order)

Three **contiguous** thirds of the transaction stream: early / middle / late.

In [7]:
n = len(df)
i1, i2 = n // 3, 2 * n // 3
D1 = df.iloc[:i1].copy()
D2 = df.iloc[i1:i2].copy()
D3 = df.iloc[i2:].copy()

FEATURE_COLS = [c for c in df.columns if c.startswith("V")] + ["Amount"]
TARGET = "Class"

def window_stats(name, w):
    return {
        "window": name,
        "n_rows": len(w),
        "fraud_count": int(w[TARGET].sum()),
        "fraud_rate": float(w[TARGET].mean()),
        "time_min": float(w["Time"].min()),
        "time_max": float(w["Time"].max()),
    }

split_summary = pd.DataFrame(
    [window_stats("D1 (baseline)", D1), window_stats("D2 (monitor)", D2), window_stats("D3 (forward eval)", D3)]
)
display(split_summary)

,window,n_rows,fraud_count,fraud_rate,time_min,time_max
0,D1 (baseline),94935,217,0.002286,0.0,65096.0
1,D2 (monitor),94936,153,0.001612,65096.0,128593.0
2,D3 (forward eval),94936,122,0.001285,128593.0,172792.0


## 2. Drift layer: D1 vs D2 (aligned with platform idea: PSI + KS)

- **PSI**: binned proportions, bin edges from **D1** quantiles (same spirit as `numerical_psi` in the repo).
- **KS**: two-sample statistic + p-value per feature.

In [8]:
EPS = 1e-6


def psi_bin_counts(ref_counts, cur_counts):
    e = ref_counts.astype(float)
    a = cur_counts.astype(float)
    if e.sum() <= 0 or a.sum() <= 0:
        return 0.0
    e = e / e.sum()
    a = a / a.sum()
    e = np.clip(e, EPS, 1.0)
    a = np.clip(a, EPS, 1.0)
    return float(np.sum((a - e) * np.log(a / e)))


def psi_for_feature(ref_ser, cur_ser, n_bins=10):
    ref = ref_ser.dropna().astype(float)
    cur = cur_ser.dropna().astype(float)
    if len(ref) < 20 or len(cur) < 20:
        return np.nan
    q = np.linspace(0, 1, n_bins + 1)
    edges = np.unique(np.quantile(ref, q, method="linear"))
    if len(edges) < 2:
        return 0.0
    rc, _ = np.histogram(ref, bins=edges)
    cc, _ = np.histogram(cur, bins=edges)
    return psi_bin_counts(rc, cc)


rows = []
for col in FEATURE_COLS:
    r, c = D1[col], D2[col]
    ks_stat, ks_p = stats.ks_2samp(r.dropna(), c.dropna())
    rows.append(
        {
            "feature": col,
            "psi_d1_vs_d2": psi_for_feature(r, c),
            "ks_statistic": float(ks_stat),
            "ks_pvalue": float(ks_p),
            "ks_significant_005": bool(ks_p < 0.05),
        }
    )

drift_table = pd.DataFrame(rows).sort_values("psi_d1_vs_d2", ascending=False)
display(drift_table.head(15))

drift_summary = {
    "n_features_high_psi_gt_0_2": int((drift_table["psi_d1_vs_d2"] > 0.2).sum()),
    "n_features_ks_sig_005": int(drift_table["ks_significant_005"].sum()),
    "mean_psi": float(drift_table["psi_d1_vs_d2"].mean()),
}
print(json.dumps(drift_summary, indent=2))

,feature,psi_d1_vs_d2,ks_statistic,ks_pvalue,ks_significant_005
2,V3,0.342230,0.238875,0.000000e+00,True
27,V28,0.226277,0.166268,0.000000e+00,True
24,V25,0.099910,0.137416,0.000000e+00,True
0,V1,0.094645,0.203979,0.000000e+00,True
21,V22,0.086742,0.098159,0.000000e+00,True
4,V5,0.086355,0.129578,0.000000e+00,True
10,V11,0.058811,0.092834,0.000000e+00,True
25,V26,0.057341,0.044252,3.264068e-81,True
22,V23,0.050692,0.087961,5.311206e-320,True
14,V15,0.044856,0.096650,0.000000e+00,True


{
  "n_features_high_psi_gt_0_2": 2,
  "n_features_ks_sig_005": 29,
  "mean_psi": 0.05116902285608287
}


## 3. Models & metrics

- **Baseline model:** trained on **D1** only → test on **D2** and **D3** (performance drop over time).
- **Adapted model:** trained on **D1 ∪ D2** → test on **D3** only (forward window, not seen in training).

Classifier: **scaled logistic regression** with `class_weight='balanced'` (strong baseline for imbalanced fraud).

Metrics: **ROC-AUC**, **PR-AUC** (average precision), **F1**, precision, recall, accuracy.

In [9]:
def Xy(frame):
    X = frame[FEATURE_COLS].astype(np.float64)
    y = frame[TARGET].astype(int)
    return X, y


def eval_model(pipe, X, y, name):
    proba = pipe.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "eval_set": name,
        "roc_auc": float(roc_auc_score(y, proba)),
        "pr_auc": float(average_precision_score(y, proba)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "accuracy": float(accuracy_score(y, pred)),
    }


def make_pipeline():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RNG,
                    solver="lbfgs",
                ),
            ),
        ]
    )


X1, y1 = Xy(D1)
X2, y2 = Xy(D2)
X3, y3 = Xy(D3)

# Baseline: train D1 only
base = make_pipeline()
base.fit(X1, y1)

# Adapted: train D1 + D2
D12 = pd.concat([D1, D2], axis=0, ignore_index=True)
X12, y12 = Xy(D12)
adapted = make_pipeline()
adapted.fit(X12, y12)

results = [
    {**eval_model(base, X2, y2, "D2 | trained D1 only"), "model": "trained_D1"},
    {**eval_model(base, X3, y3, "D3 | trained D1 only"), "model": "trained_D1"},
    {**eval_model(adapted, X3, y3, "D3 | trained D1+D2"), "model": "trained_D1+D2"},
]

metrics_df = pd.DataFrame(results)
display(metrics_df)

,eval_set,roc_auc,pr_auc,f1,precision,recall,accuracy,model
0,D2 | trained D1 only,0.883833,0.668181,0.131579,0.071813,0.784314,0.983315,trained_D1
1,D3 | trained D1 only,0.948602,0.609014,0.122249,0.066050,0.819672,0.984874,trained_D1
2,D3 | trained D1+D2,0.977575,0.745768,0.094025,0.049675,0.877049,0.978280,trained_D1+D2


## 4. One-line thesis takeaway

Compare **PR-AUC on D3**: adapted (D1+D2) vs baseline (D1 only) — forward-time generalization after including the middle window in training.

In [10]:
p_d3_base = metrics_df.loc[
    (metrics_df["model"] == "trained_D1") & (metrics_df["eval_set"].str.startswith("D3")), "pr_auc"
].values[0]
p_d3_adapt = metrics_df.loc[
    (metrics_df["model"] == "trained_D1+D2") & (metrics_df["eval_set"].str.startswith("D3")), "pr_auc"
].values[0]
delta = p_d3_adapt - p_d3_base
print(f"PR-AUC on D3: baseline (train D1) = {p_d3_base:.4f}")
print(f"PR-AUC on D3: adapted (train D1+D2) = {p_d3_adapt:.4f}")
print(f"Delta PR-AUC on D3 = {delta:+.4f}")

PR-AUC on D3: baseline (train D1) = 0.6090
PR-AUC on D3: adapted (train D1+D2) = 0.7458
Delta PR-AUC on D3 = +0.1368


## 5. (B) Export CSVs for the main platform

Saves **`D1.csv`**, **`D2.csv`**, **`D3.csv`** under `EXPORT_DIR`.



In [11]:
EXPORT_DIR = Path("fraud_temporal_splits")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

D1.to_csv(EXPORT_DIR / "D1.csv", index=False)
D2.to_csv(EXPORT_DIR / "D2.csv", index=False)
D3.to_csv(EXPORT_DIR / "D3.csv", index=False)

print("Written:", [str(p) for p in EXPORT_DIR.glob("*.csv")])

if IN_COLAB:
    from google.colab import files

    for name in ["D1.csv", "D2.csv", "D3.csv"]:
        files.download(str(EXPORT_DIR / name))

Written: ['fraud_temporal_splits/D3.csv', 'fraud_temporal_splits/D2.csv', 'fraud_temporal_splits/D1.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Optional: save tables for the thesis (CSV)

Import `split_summary.csv`, `drift_table.csv`, `metrics_df.csv` into Word/LaTeX.

In [12]:
split_summary.to_csv(EXPORT_DIR / "split_summary.csv", index=False)
drift_table.to_csv(EXPORT_DIR / "drift_d1_vs_d2.csv", index=False)
metrics_df.to_csv(EXPORT_DIR / "model_metrics.csv", index=False)
print("Saved thesis tables to", EXPORT_DIR)

Saved thesis tables to fraud_temporal_splits
